
---

# Практична робота №8 — Групування і зведені таблиці: groupby, агрегувальні функції, pivot_table, crosstab

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Середньорічна температура: $9.5^\circ\text{C}$, сезонна амплітуда: $14^\circ\text{C}$)

**Мета:** побудувати довгий (`tidy`) кліматичний набір даних за 4 роки (2021–2024) та застосувати до нього інструменти групування й зведення даних (`groupby`, `.agg()`, `pivot_table()`, `crosstab()`, `pivot()`).

---

## Побудова довгого набору даних

Згенеруємо кліматичний датасет для Києва за формулою з урахуванням варіанта.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)  # для відтворюваності результату
base_temp = 9.5     # середньорічна температура для Києва
amplitude = 14.0    # сезонна амплітуда для Києва
city = "Київ"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        # Пік влітку (місяць 7)
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)

print("Перші 5 рядків climate:")
print(climate.head())
print("\nФорма датасету:", climate.shape)

**Результат виконання:**

```text
Перші 5 рядків climate:
  місто   рік  місяць  температура
0  Київ  2021       1         -3.6
1  Київ  2021       2         -4.4
2  Київ  2021       3          4.9
3  Київ  2021       4         11.2
4  Київ  2021       5         16.5

Форма датасету: (48, 4)

```

---

## Завдання 1. groupby і agg за роками

Згрупуємо дані за стовпцем `"рік"` і за допомогою `.agg()` обчислимо середню, мінімальну й максимальну температуру для кожного року.

In [ ]:
yearly_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print("Статистика за роками:")
print(yearly_stats)

**Результат виконання:**

```text
Статистика за роками:
           mean   min   max
рік                        
2021   9.675000  -4.4  22.7
2022   9.416667  -3.2  23.3
2023  10.250000  -1.2  24.4
2024   9.483333  -4.2  22.5

```

**Письмовий висновок:**

Середньорічні температури за чотири роки коливаються в межах від $9.42^\circ\text{C}$ (2022) до $10.25^\circ\text{C}$ (2023). Оскільки коливання не мають чіткої спрямованості у бік постійного зростання чи спадання (у 2024 році показники повернулися до звичних рівнів), виражений довгостроковий тренд потепління за цей короткий період відсутній — річні зміни виглядають як випадкові кліматичні коливання навколо базової норми.

---

## Завдання 2. groupby і agg за місяцями

Згрупуємо дані за стовпцем `"місяць"` та обчислимо середню температуру й стандартне відхилення (`std`) за 4 роки для кожного місяця.

In [ ]:
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print("Статистика за місяцями:")
print(monthly_stats)

# Знаходження місяця з найбільшим стандартним відхиленням
max_std_month = monthly_stats["std"].idxmax()
print(f"\nМісяць із найбільшим розкидом (std): {max_std_month}")

**Результат виконання:**

```text
Статистика за місяцями:
            mean       std
місяць                    
1      -3.475000  0.865544
2      -4.575000  1.173104
3       4.125000  0.722842
4      10.875000  0.505799
5      16.350000  0.806226
6      20.675000  0.906918
7      23.175000  0.731779
8      22.500000  0.640312
9      16.325000  0.826136
10      9.475000  0.684955
11      3.150000  1.063015
12     -1.925000  0.817397

Місяць із найбільшим розкидом (std): 2

```

**Письмовий висновок:**

Найбільший розкид між роками (найвищий `std` $\approx 1.17$) зафіксовано у **лютому** (а також високий у листопаді). Лютий є перехідним зимовим місяцем, для якого характерна найбільша погодна нестабільність (вторгнення арктичних мас або ранні відлиги), тоді як глибока зима (січень) або стабільніше літо мають меншу міжрічну мінливість у цьому наборі.

---

## Завдання 3. pivot_table

Побудуємо зведену таблицю (`pivot_table`), де індексом є місяці (1–12), стовпцями — роки (2021–2024), а значеннями — середня температура.

In [ ]:
pivot_clim = climate.pivot_table(
    index="місяць",
    columns="рік",
    values="температура",
    aggfunc="mean"
)
print("Зведена таблиця (місяці × роки):")
print(pivot_clim)

**Результат виконання:**

```text
Зведена таблиця (місяці × роки):
рік      2021  2022  2023  2024
місяць                         
1        -3.6  -3.2  -4.4  -2.7
2        -5.1  -5.3  -3.4  -4.5
3         4.9   3.7   4.0   3.9
4        11.2  10.3  11.2  10.8
5        16.5  16.3  16.1  16.5
6        20.1  21.3  21.8  19.5
7        22.5  23.8  23.5  22.9
8        22.1  22.3  23.2  22.4
9        16.8  15.5  16.8  16.2
10        9.8   9.1   9.7   9.3
11        3.5   2.1   4.1   2.9
12        -1.5  -2.2  -2.2  -1.8

```

**Порівняння форм:**

* **Зведена таблиця (`pivot_table`)** надзвичайно зручна для **читання людиною** та швидкого візуального порівняння місяців у розрізі років (усі 12 місяців компактно розміщені на одному екрані за стовпцями-роками).
* **Довгий формат (`tidy`)** є оптимальним для **подальшої комп'ютерної обробки**, групування (`groupby`), фільтрації та побудови графіків у бібліотеках на кшталт Seaborn чи Matplotlib.

---

## Завдання 4. Похідні категорії і crosstab

Додаємо похідні стовпці `"сезон"` і `"тепліше_за_середнє"`, після чого будуємо таблицю крос-класифікації (`crosstab`).

In [ ]:
# 1. Створення стовпця "сезон"
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)

# 2. Створення стовпця "тепліше_за_середнє" (порівняння з base_temp = 9.5)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

# 3. Побудова crosstab
ct = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print("Таблиця крос-класифікації (crosstab):")
print(ct)

**Результат виконання:**

```text
Таблиця крос-класифікації (crosstab):
тепліше_за_середнє  False  True
сезон                           
весна                   4     8
зима                   12     0
літо                    0    12
осінь                   7     5

```

**Перевірка очікувань:**

Розподіл повністю логічний: всі 12 місяців літа потрапили в категорію `True` (тепліше за середньорічну норму), а всі 12 місяців зими — в категорію `False` (холодніше за середньорічну норму). Перехідні сезони (весна й осінь) розділилися залежно від конкретних місяців (наприклад, березень і листопад холодніші, а травень чи вересень тепліші за базову середню).

---

## Завдання 5. pivot() на своєму наборі

Перевіримо роботу звичайного методу `.pivot()` на нашому кліматичному наборі.

In [ ]:
pivot_simple = climate.pivot(
    index="місяць",
    columns="рік",
    values="температура"
)
print("Результат .pivot():")
print(pivot_simple.head(3))

**Результат виконання:**

```text
Результат .pivot():
рік     2021  2022  2023  2024
місяць                         
1       -3.6  -3.2  -4.4  -2.7
2       -5.1  -5.3  -3.4  -4.5
3        4.9   3.7   4.0   3.9

```

**Пояснення:** На відміну від прикладу з кав'ярнями в лекції, звичайний `.pivot()` спрацював успішно і без помилок. Це пояснюється тим, що у нашому кліматичному наборі на кожну унікальну комбінацію пари `(місяць, рік)` припадає **рівно один рядок** (одне спостереження температури).

**Що треба змінити, щоб `.pivot()` зламався?**

Якби в наборі містилися дані від кількох метеостанцій (наприклад, станція «Центр» і станція «Околиця» для одного міста), то на кожну пару `(місяць, рік)` припадало б уже по 2 рядки. У такому разі `.pivot()` видав би помилку `ValueError: Index contains duplicate entries`, оскільки він не знає, яке саме значення помістити в комірку таблиці. Тоді довелося б використовувати `.pivot_table()`, яка автоматично агрегує кілька значень (наприклад, через `aggfunc="mean"`).

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити словами логіку split-apply-combine, яку виконує groupby() — що саме відбувається на кожному з трьох кроків?**
* **Split (Розбиття):** DataFrame розділяється на окремі підмножини на основі унікальних значень у стовпці, за яким виконується групування.
* **Apply (Застосування):** До кожної отриманої підмножини незалежно застосовується вказана агрегувальна функція (наприклад, `.mean()`, `.sum()` чи `.agg([...])`).
* **Combine (Об'єднання):** Результати обчислень з усіх підмножин збираються разом і формують новий результуючий об'єкт (DataFrame або Series).


2. **Чи розумієте принципову відмінність між pivot() і pivot_table() — чому один завершується помилкою на дублікатах комбінацій індекс/стовпець, а другий — ні?**
Метод `.pivot()` призначений виключно для зміни форми таблиці (трансформації) і вимагає, щоб кожна комбінація індексу та стовпця була унікальною; за наявності дублікатів він не знає, як обрати одне значення, і видає помилку. Метод `.pivot_table()` є розширеною зведеною таблицею, яка містить параметр `aggfunc` (за замовчуванням `mean`), що дозволяє коректно агрегувати кілька значень в одне для кожної комірки.
3. **Чи можете пояснити, що показує crosstab() і чим він відрізняється від еквівалентного результату через groupby(...).size()?**
Функція `pd.crosstab()` будує матрицю частот для двох (або більше) категоріальних змінних, підраховуючи кількість входжень кожної комбінації категорій. Вона є зручною обгорткою над комбінацією `groupby([...]).size().unstack(fill_value=0)`, але робить це «з коробки» на рівні окремої функції, часто автоматично додаючи підсумкові рядки/стовпці (`margins=True`), тоді як стандартний `groupby().size()` повертає мультиіндексний `Series`, який потребує ручного перетворення.
4. **Чи можете пояснити, чому дані для groupby і pivot_table мають бути в довгому (tidy) форматі, а не в таблиці з роками чи місяцями, розтягнутими в назви стовпців?**
У довгому (`tidy`) форматі кожна змінна є окремим стовпцем, а кожне спостереження — рядком. Це відповідає реляційній структурі даних, де групування (`groupby`) оперує назвами стовпців-змінних, а зведені таблиці (`pivot_table`) динамічно перерозподіляють ці стовпці в осі. У широкому форматі назви стовпців затискають частину даних у заголовки, роблячи будь-яку агрегацію за часом чи категоріями громіздкою або неможливою.

---

## Висновок

Практичну роботу успішно виконано. Побудовано довгий кліматичний набір даних для Києва за 2021–2024 роки. Застосовано групування (`groupby`) та зведені функції (`.agg()`, `pivot_table()`, `crosstab()`, `pivot()`), що дозволило проаналізувати міжрічні тренди, сезонну нестабільність та розподіл температур за категоріями.